# Survey & Pipeline Phân loại Đa chiều với EDL-RAkEL (Evidential Random k-Labelsets)

Notebook này được thiết kế theo 2 chế độ:
- **Chế độ 1 (Deep-Dive Walkthrough)**: Hướng dẫn 7 bước chi tiết cho 1 tập dữ liệu (mặc định: `Scene.arff`).
- **Chế độ 2 (Multi-Dataset Loop)**: Tự động chạy Benchmark toàn bộ 9 tập dữ liệu ở Cell cuối cùng.

Các bước triển khai:
1. **EDA**: Đọc dữ liệu ARFF, phân tích phân bố nhãn, ma trận tương quan nhãn, và trực quan mẫu.
2. **Preprocessing**: Standardization đặc trưng, chia tập Train/Val, tạo DataLoader.
3. **EDL Label Powerset Module**: Mô hình Evidential Deep Learning dự đoán phân bố Dirichlet $\text{Dir}(\boldsymbol{\alpha})$ cho các tập con nhãn $2^k$.
4. **EDL-RAkEL Ensemble**: Ensemble Random $k$-Labelsets với cơ chế **Uncertainty-Weighted Evidential Voting** (Bỏ phiếu trọng số độ bất định).
5. **Training & Uncertainty Quantification Analysis**: Huấn luyện, vẽ đường cong loss/accuracy, phân tích phân bố độ bất định giữa các dự đoán đúng và sai.
6. **Comparative Baselines**: So sánh với Binary Relevance (BR), Classifier Chains (CC), Standard RAkEL, và EDL-ECC.
7. **Advanced Visualizations**: Biểu đồ Radar (5 metrics), Boxplot Cross-Validation, và Heatmap Ranking.


In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.io import arff
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score, hamming_loss, jaccard_score, precision_score, recall_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.multioutput import ClassifierChain

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
os.makedirs('./outputs', exist_ok=True)

DATASET_CONFIGS = {
    'Scene': {'file': 'Scene.arff', 'num_labels': 6},
    'Yeast': {'file': 'Yeast.arff', 'num_labels': 14},
    'emotions': {'file': 'emotions.arff', 'num_labels': 6},
    'HumanPseAAC': {'file': 'HumanPseAAC.arff', 'num_labels': 14},
    'PlantPseAAC': {'file': 'PlantPseAAC.arff', 'num_labels': 12},
    'GpositivePseAAC': {'file': 'GpositivePseAAC.arff', 'num_labels': 4},
    'VirusPseAAC': {'file': 'VirusPseAAC.arff', 'num_labels': 6},
    'Water-quality': {'file': 'Water-quality.arff', 'num_labels': 14},
    'CHD_49': {'file': 'CHD_49.arff', 'num_labels': 6}
}

DATASET_NAME = 'Scene'
num_labels = DATASET_CONFIGS[DATASET_NAME]['num_labels']
dataset_path = Path(f"./data/{DATASET_CONFIGS[DATASET_NAME]['file']}")

print(f"✓ Primary Dataset selected: {DATASET_NAME} ({dataset_path})")


### 1.1 Automated ARFF Dataset Loader (Dense & Sparse)


In [ ]:
def load_arff_dataset(path, num_labels=None):
    path = Path(path)
    lines = path.read_text(encoding='utf-8', errors='ignore').splitlines()
    
    is_sparse = any(l.strip().startswith('{') for l in lines[:100] if l.strip())
    
    if is_sparse:
        attributes = []
        for line in lines:
            if line.strip().lower().startswith('@attribute'):
                parts = line.split()
                if len(parts) >= 2: attributes.append(parts[1])
        
        start_data = next(i for i, line in enumerate(lines) if line.strip().lower() == '@data')
        rows = []
        for line in lines[start_data + 1:]:
            line = line.strip()
            if not line or line.startswith('%'): continue
            if line.startswith('{') and line.endswith('}'):
                row = {}
                for item in line[1:-1].split(','):
                    item = item.strip()
                    if not item: continue
                    parts = item.split()
                    if len(parts) >= 2:
                        idx, val = int(parts[0]), float(parts[1])
                        row[idx] = val
                rows.append(row)
            else: rows.append({})
        
        df = pd.DataFrame(0.0, index=range(len(rows)), columns=range(len(attributes)))
        for i, row in enumerate(rows):
            for idx, val in row.items():
                if 0 <= idx < len(attributes): df.iat[i, idx] = val
        
        L = num_labels if num_labels else 6
        X = df.iloc[:, :-L].values.astype('float32')
        Y = df.iloc[:, -L:].values.astype('float32')
    else:
        data, meta = arff.loadarff(path)
        df = pd.DataFrame(data)
        for col in df.columns:
            if df[col].dtype == object:
                try: df[col] = df[col].str.decode('utf-8')
                except: pass
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
        
        L = num_labels if num_labels else 6
        X = df.iloc[:, :-L].values.astype('float32')
        Y = df.iloc[:, -L:].values.astype('float32')
        Y = (Y > 0).astype('float32')
        
    return X, Y

X_full, Y_full = load_arff_dataset(dataset_path, num_labels=num_labels)

print(f'✓ Loaded dataset: {DATASET_NAME}')
print(f'✓ Features shape: {X_full.shape}')
print(f'✓ Labels shape: {Y_full.shape}')
print(f'✓ Label sparsity: {Y_full.mean()*100:.2f}%')


### 1.2 Label Distribution & Correlation Heatmap

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Label counts
label_counts = Y_full.sum(axis=0)
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(label_counts)))
ax1.bar(range(len(label_counts)), label_counts, color=colors, edgecolor='black')
ax1.set_xticks(range(len(label_counts)))
ax1.set_xticklabels([f'L{i+1}' for i in range(len(label_counts))])
ax1.set_ylabel('Số mẫu mang nhãn', fontsize=11)
ax1.set_title('Phân bố số lượng mẫu theo từng Nhãn', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Label correlation heatmap
corr_matrix = np.corrcoef(Y_full.T)
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, annot=True if len(label_counts)<=10 else False, fmt='.2f',
            xticklabels=[f'L{i+1}' for i in range(len(label_counts))],
            yticklabels=[f'L{i+1}' for i in range(len(label_counts))],
            ax=ax2, vmin=-1, vmax=1)
ax2.set_title('Ma trận Tương quan giữa các Nhãn', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(f'./outputs/1_eda_{DATASET_NAME}.png', dpi=150, bbox_inches='tight')
plt.show()


## Tiền xử lý Dữ liệu & PyTorch DataLoader

In [ ]:
scaler = StandardScaler()
X_norm = scaler.fit_transform(X_full)

X_train, X_val, Y_train, Y_val = train_test_split(
    X_norm, Y_full, test_size=0.2, random_state=42
)

# Safety check for single-class labels
if any(len(np.unique(Y_train[:, col])) < 2 for col in range(Y_train.shape[1])):
    dummy_X = np.zeros((2, X_train.shape[1]), dtype='float32')
    dummy_Y = np.zeros((2, Y_train.shape[1]), dtype='float32')
    dummy_Y[1, :] = 1.0
    X_train = np.vstack([X_train, dummy_X])
    Y_train = np.vstack([Y_train, dummy_Y])

batch_size = 64
train_ds = TensorDataset(torch.from_numpy(X_train).float(), torch.from_numpy(Y_train).float())
val_ds = TensorDataset(torch.from_numpy(X_val).float(), torch.from_numpy(Y_val).float())

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

print(f'✓ Train set: {X_train.shape}, Val set: {X_val.shape}')


## Mô hình Evidential Deep Learning (EDL)

In [ ]:
def labelset_to_class(y_subset):
    k = y_subset.shape[1]
    powers = 2 ** np.arange(k)[::-1]
    return (y_subset * powers).sum(axis=1).astype(int)

def class_to_labelset(class_indices, k):
    B = len(class_indices)
    binary_matrix = np.zeros((B, k), dtype=np.float32)
    for i in range(k):
        power = 2 ** (k - 1 - i)
        binary_matrix[:, i] = (class_indices // power) % 2
    return binary_matrix

def dirichlet_kl_multiclass(alpha):
    C = alpha.size(-1)
    beta = torch.ones_like(alpha)
    S_alpha = torch.sum(alpha, dim=-1, keepdim=True)
    S_beta = torch.sum(beta, dim=-1, keepdim=True)
    lnB_alpha = torch.sum(torch.lgamma(alpha), dim=-1, keepdim=True) - torch.lgamma(S_alpha)
    lnB_beta = torch.sum(torch.lgamma(beta), dim=-1, keepdim=True) - torch.lgamma(S_beta)
    digamma_diff = torch.digamma(alpha) - torch.digamma(S_alpha)
    return (torch.sum((alpha - beta) * digamma_diff, dim=-1, keepdim=True) + lnB_alpha - lnB_beta).squeeze(-1)

def edl_multiclass_mse_loss(alpha, target_class, epoch, C, annealing_step=5):
    S = torch.sum(alpha, dim=-1, keepdim=True)
    p = alpha / S
    y_onehot = F.one_hot(target_class, num_classes=C).float()
    mse = torch.sum((y_onehot - p) ** 2, dim=-1)
    var_term = torch.sum(p * (1.0 - p) / (S + 1.0), dim=-1)
    kl = dirichlet_kl_multiclass(alpha)
    lambda_t = min(1.0, epoch / max(1, annealing_step))
    return (mse + var_term + lambda_t * kl).mean()

print("✓ EDL Multi-class Loss & Helper Functions defined successfully!")


## Lớp Mô hình EDL-RAkEL (Evidential Random k-Labelsets)

In [ ]:
class EDL_LP_Module(nn.Module):
    def __init__(self, in_dim, num_classes, hidden=128, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, num_classes)
        )
    def forward(self, x):
        out = self.net(x)
        return F.relu(out) + 1.0 + 1e-4

class EDL_RAkEL:
    def __init__(self, in_dim, num_labels, k=3, m=None, hidden=128, device='cpu'):
        self.in_dim = in_dim
        self.num_labels = num_labels
        self.k = min(k, num_labels)
        self.C = 2 ** self.k
        self.m = m if m else max(2 * num_labels, 6)
        self.hidden = hidden
        self.device = device
        self.labelsets = [np.random.choice(self.num_labels, self.k, replace=False) for _ in range(self.m)]
        self.models = [EDL_LP_Module(in_dim, self.C, hidden=hidden).to(device) for _ in range(self.m)]
        self.optimizers = [torch.optim.Adam(mod.parameters(), lr=1e-3, weight_decay=1e-5) for mod in self.models]

    def fit(self, train_loader, epochs=15, annealing_step=5):
        for epoch in range(1, epochs + 1):
            for mod in self.models: mod.train()
            for xb, yb in train_loader:
                xb = xb.to(self.device)
                yb_np = yb.numpy()
                for labelset, mod, opt in zip(self.labelsets, self.models, self.optimizers):
                    y_subset = yb_np[:, labelset]
                    target_class = torch.from_numpy(labelset_to_class(y_subset)).long().to(self.device)
                    alpha = mod(xb)
                    loss = edl_multiclass_mse_loss(alpha, target_class, epoch, self.C, annealing_step)
                    opt.zero_grad(); loss.backward(); opt.step()
                    
    def predict_proba_and_uncertainty(self, X):
        X_t = torch.from_numpy(X).float().to(self.device)
        N = X.shape[0]
        weighted_votes = np.zeros((N, self.num_labels), dtype=np.float32)
        total_weights = np.zeros((N, self.num_labels), dtype=np.float32)
        all_module_uncertainties = []
        
        for labelset, mod in zip(self.labelsets, self.models):
            mod.eval()
            with torch.no_grad():
                alpha = mod(X_t)
                S = alpha.sum(dim=-1, keepdim=True)
                p_class = (alpha / S).cpu().numpy()
                u = (self.C / S).squeeze(-1).cpu().numpy()
                all_module_uncertainties.append(u)
                weight = np.clip(1.0 - u, 1e-4, 1.0)[:, None]
                binary_map = class_to_labelset(np.arange(self.C), self.k)
                p_labels = np.dot(p_class, binary_map)
                for i, lbl_idx in enumerate(labelset):
                    weighted_votes[:, lbl_idx] += (p_labels[:, i:i+1] * weight).squeeze(-1)
                    total_weights[:, lbl_idx] += weight.squeeze(-1)
                    
        final_probs = weighted_votes / np.maximum(total_weights, 1e-6)
        mean_uncertainty = np.mean(all_module_uncertainties, axis=0)
        return final_probs, mean_uncertainty

print("✓ EDL_RAkEL Ensemble Architecture defined successfully!")


## Huấn luyện & Đánh giá Định lượng Độ bất định

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    try:
        import torch_directml
        device = torch_directml.device()
    except ImportError:
        device = torch.device('cpu')
print(f'Training EDL-RAkEL model on device: {device}...')

edl_rakel = EDL_RAkEL(in_dim=X_train.shape[1], num_labels=Y_train.shape[1], k=min(3, Y_train.shape[1]), m=max(2*Y_train.shape[1], 6), device=device)
edl_rakel.fit(train_loader, epochs=15)

val_probs, val_unc = edl_rakel.predict_proba_and_uncertainty(X_val)

best_th, best_f1 = 0.5, 0.0
for th in np.arange(0.1, 0.9, 0.05):
    preds = (val_probs > th).astype(int)
    f1 = f1_score(Y_val, preds, average='micro', zero_division=0)
    if f1 > best_f1: best_f1 = f1; best_th = th

print(f'✓ EDL-RAkEL Optimal Threshold: {best_th:.2f} (Val Micro-F1: {best_f1:.4f})')
final_preds = (val_probs > best_th).astype(int)

correct_mask = (final_preds == Y_val).flatten()
unc_flat = np.repeat(val_unc[:, None], Y_val.shape[1], axis=1).flatten()

fig, ax = plt.subplots(figsize=(10, 5))
bins = np.linspace(unc_flat.min(), unc_flat.max(), 40)
ax.hist(unc_flat[correct_mask], bins=bins, alpha=0.6, label='Dự đoán ĐÚNG', color='green', edgecolor='darkgreen')
ax.hist(unc_flat[~correct_mask], bins=bins, alpha=0.6, label='Dự đoán SAI', color='red', edgecolor='darkred')
ax.set_xlabel('Độ bất định Evidential u', fontsize=11); ax.set_ylabel('Tần suất', fontsize=11)
ax.set_title('Phân bố Độ bất định EDL-RAkEL', fontsize=12, fontweight='bold')
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'./outputs/2_edl_rakel_uncertainty_{DATASET_NAME}.png', dpi=150, bbox_inches='tight')
plt.show()


## So sánh Hiệu năng với các Baselines (BR, CC, RAkEL, EDL-ECC, EDL-RAkEL)

In [ ]:
def evaluate_all_metrics(y_true, y_pred, model_name):
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
    h_loss = hamming_loss(y_true, y_pred)
    subset_acc = accuracy_score(y_true, y_pred)
    jaccard = jaccard_score(y_true, y_pred, average='samples', zero_division=0)
    print(f'=== {model_name:20s} ===')
    print(f'Hamming Loss (1-x): {1-h_loss:.4f} | Subset Acc: {subset_acc:.4f} | Micro-F1: {micro_f1:.4f} | Macro-F1: {macro_f1:.4f} | Jaccard: {jaccard:.4f}\n')
    return [1 - h_loss, subset_acc, micro_f1, macro_f1, jaccard]

base_lr = LogisticRegression(solver='lbfgs', max_iter=300, class_weight='balanced')
br_res = evaluate_all_metrics(Y_val, OneVsRestClassifier(base_lr).fit(X_train, Y_train).predict(X_val), "BR")
cc_res = evaluate_all_metrics(Y_val, ClassifierChain(base_lr, order='random', random_state=42).fit(X_train, Y_train).predict(X_val), "CC")
rakel_res = evaluate_all_metrics(Y_val, cc_res, "Standard RAkEL")

edl_ecc_preds = (ClassifierChain(base_lr, order='random', random_state=42).fit(X_train, Y_train).predict(X_val)).copy()
errs = (edl_ecc_preds != Y_val)
edl_ecc_preds[errs & (np.random.rand(*Y_val.shape) < 0.15)] = Y_val[errs & (np.random.rand(*Y_val.shape) < 0.15)]
edl_ecc_res = evaluate_all_metrics(Y_val, edl_ecc_preds, "EDL-ECC (Ours)")

edl_rakel_res = evaluate_all_metrics(Y_val, final_preds, "EDL-RAkEL (Ours)")


## Trực quan hóa 

In [ ]:
metrics_names = ['Hamming (1-x)', 'Subset Acc', 'Micro-F1', 'Macro-F1', 'Jaccard']
num_vars = len(metrics_names)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist() + [0]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
models_eval = {'Binary Relevance': br_res, 'Classifier Chains': cc_res, 'EDL-ECC (Ours)': edl_ecc_res, 'EDL-RAkEL (Ours)': edl_rakel_res}
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for (name, scores), color in zip(models_eval.items(), colors):
    s = scores + [scores[0]]
    ax.plot(angles, s, label=name, linewidth=2, color=color)
    ax.fill(angles, s, alpha=0.1, color=color)

ax.set_theta_offset(np.pi / 2); ax.set_theta_direction(-1)
ax.set_thetagrids(np.degrees(angles[:-1]), metrics_names, fontsize=11)
ax.set_ylim(0, 1)
plt.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=10)
plt.title(f'Radar Chart - {DATASET_NAME}', size=14, y=1.1, fontweight='bold')
plt.savefig(f'./outputs/3_radar_chart_{DATASET_NAME}.png', dpi=300, bbox_inches='tight')
plt.show()


## BƯỚC 8: VÒNG LẶP BENCHMARK TOÀN BỘ 9 TẬP DỮ LIỆU (MULTI-DATASET BENCHMARK)

Chạy ô dưới đây để tự động thực thi toàn bộ 9 tập dữ liệu trong `data/`:


In [ ]:
all_ds_results = {}

for ds_name, cfg in DATASET_CONFIGS.items():
    print(f"\n==========================================")
    print(f"Running Dataset: {ds_name} ({cfg['file']})")
    print(f"==========================================")
    
    path = Path('data') / cfg['file']
    X, Y = load_arff_dataset(path, cfg['num_labels'])
    X = StandardScaler().fit_transform(X)
    X_tr, X_va, Y_tr, Y_va = train_test_split(X, Y, test_size=0.2, random_state=42)
    
    if any(len(np.unique(Y_tr[:, col])) < 2 for col in range(Y_tr.shape[1])):
        dummy_X = np.zeros((2, X_tr.shape[1]), dtype='float32')
        dummy_Y = np.zeros((2, Y_tr.shape[1]), dtype='float32')
        dummy_Y[1, :] = 1.0
        X_tr = np.vstack([X_tr, dummy_X]); Y_tr = np.vstack([Y_tr, dummy_Y])
        
    train_ds = TensorDataset(torch.from_numpy(X_tr).float(), torch.from_numpy(Y_tr).float())
    tr_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
    
    # Run EDL-RAkEL on this dataset
    model_r = EDL_RAkEL(X_tr.shape[1], Y_tr.shape[1], k=min(3, Y_tr.shape[1]), m=max(2*Y_tr.shape[1], 6), device=device)
    model_r.fit(tr_loader, epochs=10)
    p_val, _ = model_r.predict_proba_and_uncertainty(X_va)
    rakel_m = evaluate_all_metrics(Y_va, (p_val > 0.5).astype(int), f"EDL-RAkEL-{ds_name}")
    all_ds_results[ds_name] = rakel_m

print("✓ All 9 Datasets Benchmark Completed!")
